# State Management in Async, Multi-Step Agent Workflows
### A runnable LangGraph tour (LangGraph 1.x)

Every concept from the "5 rules of state management" is demonstrated in its own section — **no LLM API keys needed**: the LLM and tools are deterministic mocks, so the focus stays on the state mechanics.

| # | Concept | Section |
|---|---------|---------|
| 1 | State schema & reducers (single source of truth) | 1 |
| 2 | Checkpointing — externalized, durable state | 2, 3 |
| 3 | Thread / session isolation | 3 |
| 4 | Explicit state machine over implicit control flow | 4 |
| 5 | Human-in-the-loop as a persisted *waiting state* | 5 |
| 6 | Crash recovery — resume after failure | 6 |
| 7 | Idempotency + retries for side effects | 7 |
| 8 | Parallel fan-out & merge conflicts | 8 |
| 9 | Time travel — replay & fork | 9 |
| 10 | Context-window pruning (RemoveMessage) | 10 |
| 11 | Async execution (`ainvoke` / `astream`) | 11 |

> **Mental model:** a LangGraph run advances in *supersteps*. Each superstep executes one or more nodes in parallel, nodes return **partial state updates**, reducers merge them, and (with a checkpointer) the merged result is **checkpointed** before the next superstep begins.

## 1. State schema & reducers — the workflow's single source of truth

A LangGraph state is a `TypedDict`. The **reducer** attached to each field decides how a node's partial update merges into the existing value:

- **No reducer** → the update **overwrites** the field.
- **`Annotated[list[str], operator.add]`** → the update is **appended** (concatenated) — perfect for an append-only **event log** (audit trail).
- **`Annotated[list, add_messages]`** → LangGraph's built-in message reducer: assigns IDs, dedupes, and supports removals (used in §10).

Rule #1 of the playbook: **never hold workflow state in process memory** — the state schema *is* the durable contract between steps.

In [1]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ---- The workflow state: externalized, typed, reducer-annotated ----
class AgentState(TypedDict):
    task: str                                        # no reducer -> overwrite
    risk: str                                        # no reducer -> overwrite
    events: Annotated[list[str], operator.add]       # append-only event log (audit trail)
    plan: str                                        # no reducer -> overwrite

def mock_llm(prompt: str) -> str:                    # deterministic fake LLM (no API key)
    return "PLAN: verify order -> check risk policy -> refund customer"

def triage(state: AgentState) -> dict:               # step 1: classify the task
    return {"task": "refund_order_42", "risk": "low", "events": ["triaged: task received"]}

def plan(state: AgentState) -> dict:                 # step 2: LLM plans next actions
    return {"plan": mock_llm(state["task"]), "events": [f"planned with LLM for {state['task']}" ]}

g = StateGraph(AgentState)
g.add_node("triage", triage)
g.add_node("plan", plan)
g.add_edge(START, "triage")
g.add_edge("triage", "plan")
g.add_edge("plan", END)

app = g.compile()                                    # no checkpointer yet — plain run
final = app.invoke({"task": "", "risk": "", "events": [], "plan": ""})
print("final state:", final)
print()
print("events accumulated across nodes (reducer = append):")
for e in final["events"]:
    print("  -", e)

# ---- Overwrite vs reducer, side by side ----
class OverwriteVsAdd(TypedDict):
    plain: int                        # overwrite semantics
    summed: Annotated[int, operator.add]  # reducer semantics

mini = StateGraph(OverwriteVsAdd)
mini.add_node("a", lambda s: {"plain": 1, "summed": 1})
mini.add_node("b", lambda s: {"plain": 1, "summed": 1})   # sequential: b runs after a
mini.add_edge(START, "a"); mini.add_edge("a", "b"); mini.add_edge("b", END)
r = mini.compile().invoke({"plain": 0, "summed": 0})
print()
print(f"overwrite field ('plain')     : {r['plain']}  (each node replaced it)")
print(f"reducer field   ('summed')    : {r['summed']}  (both nodes' contributions merged)")

final state: {'task': 'refund_order_42', 'risk': 'low', 'events': ['triaged: task received', 'planned with LLM for refund_order_42'], 'plan': 'PLAN: verify order -> check risk policy -> refund customer'}

events accumulated across nodes (reducer = append):
  - triaged: task received
  - planned with LLM for refund_order_42

overwrite field ('plain')     : 1  (each node replaced it)
reducer field   ('summed')    : 2  (both nodes' contributions merged)


## 2. Checkpointing — externalized, durable state after every step

Compile the same graph with a **checkpointer** and pass a `thread_id` in the config. LangGraph now saves a **checkpoint of the full state at every superstep** — the workflow's state lives outside any worker process.

- `InMemorySaver` — checkpoints in RAM (great for dev/tests, **lost on restart**).
- `SqliteSaver` / `PostgresSaver` — durable, survives process restarts (production).

This is what makes workers **stateless and horizontally scalable**: any worker can pick up a thread because the state isn't *in* the worker.

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

g = StateGraph(AgentState)
g.add_node("triage", triage)
g.add_node("plan", plan)
g.add_edge(START, "triage"); g.add_edge("triage", "plan"); g.add_edge("plan", END)

app = g.compile(checkpointer=InMemorySaver())        # <-- durability switch
cfg = {"configurable": {"thread_id": "order-42"}}    # <-- identity of this workflow run

app.invoke({"task": "", "risk": "", "events": [], "plan": ""}, cfg)

snapshot = app.get_state(cfg)                        # inspect durable state at any time
print("latest state values :", snapshot.values)
print("next nodes to run   :", snapshot.next)       # empty tuple = workflow finished
print()
history = list(app.get_state_history(cfg))           # full checkpoint timeline
print(f"{len(history)} checkpoints persisted (one per superstep), newest first:")
for cp in history:
    step = cp.metadata.get("step")
    wrote = {n for n in (cp.metadata.get("writes") or {})}
    print(f"  step {step}: next={cp.next} nodes_wrote={sorted(wrote)}")

latest state values : {'task': 'refund_order_42', 'risk': 'low', 'events': ['triaged: task received', 'planned with LLM for refund_order_42'], 'plan': 'PLAN: verify order -> check risk policy -> refund customer'}
next nodes to run   : ()

4 checkpoints persisted (one per superstep), newest first:
  step 2: next=() nodes_wrote=[]
  step 1: next=('plan',) nodes_wrote=[]
  step 0: next=('triage',) nodes_wrote=[]
  step -1: next=('__start__',) nodes_wrote=[]


## 3. Thread isolation & durability across "restarts"

- **Different `thread_id`s** on the same compiled graph are completely isolated conversations/workflows — this is how one deployment serves many customers.
- With `SqliteSaver`, a **brand-new graph object + fresh DB connection** can read the state written by a previous process: the checkpoint *is* the handoff between workers.

In [3]:
import sqlite3, os
from langgraph.checkpoint.sqlite import SqliteSaver

# --- isolation: same graph, two independent threads ---
app = g.compile(checkpointer=InMemorySaver())
cfg_a = {"configurable": {"thread_id": "customer-A"}}
cfg_b = {"configurable": {"thread_id": "customer-B"}}
app.invoke({"task": "", "risk": "", "events": [], "plan": ""}, cfg_a)
app.invoke({"task": "", "risk": "", "events": [], "plan": ""}, cfg_b)
print("thread A events:", app.get_state(cfg_a).values["events"])
print("thread B events:", app.get_state(cfg_b).values["events"])
print("isolated:", app.get_state(cfg_a).values["events"] == app.get_state(cfg_b).values["events"])

# --- durability: sqlite file survives a whole new 'process' ---
if os.path.exists("/tmp/agent_ck.db"): os.remove("/tmp/agent_ck.db")
db = StateGraph(AgentState)
db.add_node("triage", triage); db.add_node("plan", plan)
db.add_edge(START, "triage"); db.add_edge("triage", "plan"); db.add_edge("plan", END)

writer = db.compile(checkpointer=SqliteSaver(sqlite3.connect("/tmp/agent_ck.db", check_same_thread=False)))
wcfg = {"configurable": {"thread_id": "order-42"}}
writer.invoke({"task": "", "risk": "", "events": [], "plan": ""}, wcfg)

# simulate a process restart: brand-new connection + brand-new compiled graph
restart = db.compile(checkpointer=SqliteSaver(sqlite3.connect("/tmp/agent_ck.db", check_same_thread=False)))
print()
print("after 'restart', recovered state from disk:", restart.get_state(wcfg).values)

thread A events: ['triaged: task received', 'planned with LLM for refund_order_42']
thread B events: ['triaged: task received', 'planned with LLM for refund_order_42']
isolated: True

after 'restart', recovered state from disk: {'task': 'refund_order_42', 'risk': 'low', 'events': ['triaged: task received', 'planned with LLM for refund_order_42'], 'plan': 'PLAN: verify order -> check risk policy -> refund customer'}


## 4. Explicit state machine — legal transitions, not if-else soup

A production agent workflow is modeled as an **explicit state machine**: named states (`TRIAGE → ASSESS → WAITING_HUMAN → EXECUTE → DONE/FAILED`) and legal transitions (edges). Async gaps (tool calls, human approvals) are just **waiting states with timeouts** — not blocked threads.

We build a realistic **refund agent**: triage the order, assess risk, then either auto-approve (low risk) or route to a human gate (high risk), then execute the refund through an **idempotent** tool (§7 shows the retry/idempotency mechanics).

```
START ──> triage ──> assess ──┬─(low risk)──> execute ──> done ──> END
                              └─(high risk)─> human_gate ──> execute ──> done ──> END
```

In [4]:
class RefundState(TypedDict):
    task: str
    risk: str
    decision: str
    events: Annotated[list[str], operator.add]

def triage_node(s):    return {"events": [f"triage: received {s['task']}"]}
def assess_node(s):    return {"risk": s["risk"], "events": [f"assess: risk={s['risk']}"]}

def route(state: RefundState) -> str:                # conditional edge = legal transition
    return "auto" if state["risk"] == "low" else "human"

def execute_node(s):
    return {"decision": "refunded", "events": [f"execute: refund issued for {s['task']}"]}

def done_node(s):      return {"events": ["done: workflow complete"]}

refund = StateGraph(RefundState)
refund.add_node("triage", triage_node)
refund.add_node("assess", assess_node)
refund.add_node("human_gate", lambda s: {"events": ["human_gate: awaiting approval"]})  # expanded in §5
refund.add_node("execute", execute_node)
refund.add_node("done", done_node)

refund.add_edge(START, "triage")
refund.add_edge("triage", "assess")
refund.add_conditional_edges("assess", route, {"auto": "execute", "human": "human_gate"})
refund.add_edge("human_gate", "execute")             # legal transition after approval
refund.add_edge("execute", "done")
refund.add_edge("done", END)

rapp = refund.compile(checkpointer=InMemorySaver())

# LOW-RISK path runs start to finish, no human needed
low = {"configurable": {"thread_id": "refund-low"}}
rapp.invoke({"task": "order-17", "risk": "low", "decision": "", "events": []}, low)
snap = rapp.get_state(low)
print("low-risk final state:", snap.values)
print("next after completion:", snap.next)
print()
for e in snap.values["events"]:
    print("  •", e)

low-risk final state: {'task': 'order-17', 'risk': 'low', 'decision': 'refunded', 'events': ['triage: received order-17', 'assess: risk=low', 'execute: refund issued for order-17', 'done: workflow complete']}
next after completion: ()

  • triage: received order-17
  • assess: risk=low
  • execute: refund issued for order-17
  • done: workflow complete


## 5. Human-in-the-loop — the async gap as a persisted *waiting state*

A human approval can take minutes, hours, or days. You must **not** block a worker that long. LangGraph's `interrupt()`:

1. Suspends graph execution at that exact point,
2. **Persists state via the checkpointer** (this is why a checkpointer is required),
3. Surfaces the interrupt payload to the caller under `__interrupt__`,
4. Waits **indefinitely** — the thread is a durable `WAITING_HUMAN` state,
5. `Command(resume=...)` re-invokes the graph; the resume value becomes the **return value of `interrupt()`** and the node continues as if nothing happened.

`app.get_state(...).next == ('human_gate',)` is your "this workflow is parked at the human gate" signal — exactly what an ops dashboard would poll.

**High-risk path** below hits the gate. Note the event log: everything before the interrupt is already checkpointed.

In [5]:
from langgraph.types import interrupt, Command

def human_gate_node(s: RefundState) -> dict:
    approval = interrupt({"question": f"Approve refund for {s['task']} (risk={s['risk']})?"})
    return {"decision": "approved", "events": [f"human_gate: human said '{approval}'"]}

# rebuild the graph cleanly with the interrupt-based gate
refund2 = StateGraph(RefundState)
refund2.add_node("triage", triage_node)
refund2.add_node("assess", assess_node)
refund2.add_node("human_gate", human_gate_node)
refund2.add_node("execute", execute_node)
refund2.add_node("done", done_node)
refund2.add_edge(START, "triage")
refund2.add_edge("triage", "assess")
refund2.add_conditional_edges("assess", route, {"auto": "execute", "human": "human_gate"})
refund2.add_edge("human_gate", "execute")
refund2.add_edge("execute", "done")
refund2.add_edge("done", END)

rapp2 = refund2.compile(checkpointer=InMemorySaver())

high = {"configurable": {"thread_id": "refund-high"}}
paused_result = rapp2.invoke({"task": "order-99", "risk": "high", "decision": "", "events": []}, high)

pending = paused_result["__interrupt__"]              # graph returned a pending interrupt
print("graph PAUSED. interrupt payload:", pending[0].value)
snap = rapp2.get_state(high)
print("parked at node:", snap.next, " <- durable WAITING_HUMAN state")
print("checkpointed events so far:", snap.values["events"])

# ...hours later, a human approves in the ops UI. Resume the SAME thread:
resumed = rapp2.invoke(Command(resume="yes"), high)
print()
print("resumed final state:", resumed)
snap2 = rapp2.get_state(high)
print("thread state now:", snap2.values)

graph PAUSED. interrupt payload: {'question': 'Approve refund for order-99 (risk=high)?'}
parked at node: ('human_gate',)  <- durable WAITING_HUMAN state
checkpointed events so far: ['triage: received order-99', 'assess: risk=high']

resumed final state: {'task': 'order-99', 'risk': 'high', 'decision': 'refunded', 'events': ['triage: received order-99', 'assess: risk=high', "human_gate: human said 'yes'", 'execute: refund issued for order-99', 'done: workflow complete']}
thread state now: {'task': 'order-99', 'risk': 'high', 'decision': 'refunded', 'events': ['triage: received order-99', 'assess: risk=high', "human_gate: human said 'yes'", 'execute: refund issued for order-99', 'done: workflow complete']}


## 6. Crash recovery — resume after failure

Because a checkpoint is written **before each node runs** and after every superstep, a node that crashes mid-workflow leaves the thread parked at the failed node with all prior progress intact.

Recovery is: **fix the cause, then `invoke(None, same_config)`** — `None` means "no new input, continue the persisted thread." No manual re-execution of earlier steps, no lost work.

In [6]:
flag = {"crash": True}

def bad_node(s: RefundState) -> dict:
    if flag["crash"]:
        raise RuntimeError("payment provider 503: connection reset")   # simulated outage
    return {"events": ["execute: provider recovered, refund issued"]}

crashy = StateGraph(RefundState)
crashy.add_node("triage", triage_node)
crashy.add_node("bad", bad_node)
crashy.add_node("done", done_node)
crashy.add_edge(START, "triage"); crashy.add_edge("triage", "bad"); crashy.add_edge("bad", "done"); crashy.add_edge("done", END)

capp = crashy.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "crash-1"}}

try:
    capp.invoke({"task": "order-7", "risk": "low", "decision": "", "events": []}, cfg)
except RuntimeError as e:
    print("workflow crashed:", e)

snap = capp.get_state(cfg)
print("durable state after crash :", snap.values)
print("parked at node            :", snap.next, " <- resume from here, triage NOT re-run")

flag["crash"] = False                                # "the outage is over" — fix and resume
recovered = capp.invoke(None, cfg)                   # None = continue persisted thread
print()
print("recovered:", recovered["events"])

workflow crashed: payment provider 503: connection reset
durable state after crash : {'task': 'order-7', 'risk': 'low', 'decision': '', 'events': ['triage: received order-7']}
parked at node            : ('bad',)  <- resume from here, triage NOT re-run

recovered: ['triage: received order-7', 'execute: provider recovered, refund issued', 'done: workflow complete']


## 7. Idempotency & retries — making re-execution harmless

Async systems **will** re-execute things: node retries, resumed threads, at-least-once queues. Two complementary layers:

1. **`RetryPolicy`** — declarative, per-node retry with exponential backoff for *transient* failures. Note: pass `retry_on=<ExceptionClass>` explicitly — by default LangGraph only retries network-ish errors, not arbitrary exceptions.
2. **Idempotency keys for side effects** — every real-world action (payment, email, DB write) checks a ledger before executing. If the step already ran, it returns the recorded result instead of double-charging the customer.

In [7]:
from langgraph.types import RetryPolicy

# ---- Layer 1: RetryPolicy for transient failures ----
attempts = {"flaky_call": 0}
def flaky_provider(s: RefundState) -> dict:
    attempts["flaky_call"] += 1
    if attempts["flaky_call"] < 3:
        raise ConnectionError("transient network blip")   # will be retried
    return {"events": [f"provider: succeeded on attempt {attempts['flaky_call']}"]}

rg = StateGraph(RefundState)
rg.add_node("flaky", flaky_provider,
            retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01,
                                     backoff_factor=1.0, jitter=0,
                                     retry_on=ConnectionError))   # <-- explicit match
rg.add_edge(START, "flaky"); rg.add_edge("flaky", END)
print("retry result :", rg.compile().invoke({"task": "", "risk": "", "decision": "", "events": []})["events"])
print("attempts made:", attempts["flaky_call"])

# ---- Layer 2: idempotency ledger for side effects ----
EXECUTION_LEDGER: dict[str, str] = {}

def idempotent_refund(step_id: str, amount: str) -> str:
    if step_id in EXECUTION_LEDGER:                       # already executed? never twice
        return f"SKIPPED (already executed): {EXECUTION_LEDGER[step_id]}"
    result = f"refunded {amount}"
    EXECUTION_LEDGER[step_id] = result
    return f"EXECUTED: {result}"

step = "refund:order-99:v1"
print()
print("first call (async retry arrives) :", idempotent_refund(step, "$42.00"))
print("duplicate call (same step id)    :", idempotent_refund(step, "$42.00"))
print("ledger:", EXECUTION_LEDGER, "-> customer charged exactly once")

retry result : ['provider: succeeded on attempt 3']
attempts made: 3

first call (async retry arrives) : EXECUTED: refunded $42.00
duplicate call (same step id)    : SKIPPED (already executed): refunded $42.00
ledger: {'refund:order-99:v1': 'refunded $42.00'} -> customer charged exactly once


## 8. Parallel fan-out & merge conflicts

A node with multiple outgoing edges fans out: all destination nodes run **in parallel in the same superstep**. The danger: if two parallel nodes update the **same reducer-less field**, the merge is ambiguous and LangGraph raises `InvalidUpdateError` — fail loudly at design time instead of silently losing writes.

Fix: give the shared field a **reducer** (`operator.add`) so both writes merge deterministically — this is exactly how you aggregate parallel agent outputs (e.g., 3 reviewers scoring a plan).

In [8]:
from langgraph.errors import InvalidUpdateError

def reviewer_a(s): return {"risk": 1}
def reviewer_b(s): return {"risk": 2}

# --- without a reducer: ambiguous parallel writes fail loudly ---
bad = StateGraph(RefundState)
bad.add_node("rev_a", reviewer_a); bad.add_node("rev_b", reviewer_b)
bad.add_edge(START, "rev_a"); bad.add_edge(START, "rev_b")     # parallel!
bad.add_edge("rev_a", END);   bad.add_edge("rev_b", END)
try:
    bad.compile().invoke({"task": "", "risk": "", "decision": "", "events": []})
    print("no conflict raised")
except InvalidUpdateError as e:
    print("InvalidUpdateError (ambiguous parallel write):", str(e)[:80], "...")

# --- with a reducer: deterministic merge ---
class AggState(TypedDict):
    task: str
    risk: str
    decision: str
    events: Annotated[list[str], operator.add]
    risk_score: Annotated[int, operator.add]      # reducer => parallel-safe

agg = StateGraph(AggState)
agg.add_node("rev_a", lambda s: {"risk_score": 1, "events": ["reviewer A scored"]})
agg.add_node("rev_b", lambda s: {"risk_score": 2, "events": ["reviewer B scored"]})
agg.add_edge(START, "rev_a"); agg.add_edge(START, "rev_b")     # parallel!
agg.add_edge("rev_a", END);   agg.add_edge("rev_b", END)
r = agg.compile().invoke({"task": "t", "risk": "", "decision": "", "events": [], "risk_score": 0})
print()
print("merged parallel result:", {"risk_score": r["risk_score"], "events": r["events"]})

InvalidUpdateError (ambiguous parallel write): At key 'risk': Can receive only one value per step. Use an Annotated key to hand ...

merged parallel result: {'risk_score': 3, 'events': ['reviewer A scored', 'reviewer B scored']}


## 9. Time travel — replay & fork from any checkpoint

Because every superstep is checkpointed, you can:

- **Replay**: re-invoke the thread from a specific `checkpoint_id` (great for debugging *"what did the agent see when it decided this?"* and for eval regression suites).
- **Fork**: `update_state()` at a historical checkpoint to patch a value (e.g., correct a bad plan), producing a **new branch** of the workflow — then continue from there.

In [9]:
tt_app = g.compile(checkpointer=InMemorySaver())       # triage -> plan graph
tt_cfg = {"configurable": {"thread_id": "timetravel"}}
tt_app.invoke({"task": "", "risk": "", "events": [], "plan": ""}, tt_cfg)

history = list(tt_app.get_state_history(tt_cfg))
print("checkpoint timeline (newest first):")
for cp in history:
    print(f"  step {cp.metadata.get('step')}: next={cp.next}")

older = history[0]                                     # checkpoint parked right after 'triage'
# NOTE: reuse the checkpoint's OWN config object — it carries the required
# 'checkpoint_ns' key; a hand-built config dict without it breaks update_state.
older_cfg = older.config
print()
print("REPLAY from step", older.metadata.get("step"), "-> re-runs 'plan' with identical inputs:")
replayed = tt_app.invoke(None, older_cfg)
print("  replayed events:", replayed["events"])

print()
print("FORK: patch the state at that checkpoint, branch, and continue:")
fork_cfg = tt_app.update_state(older_cfg, {"plan": "FORKED PLAN: escalate to fraud team instead"})
forked = tt_app.invoke(None, fork_cfg)
print("  forked branch state:", forked)

checkpoint timeline (newest first):
  step 2: next=()
  step 1: next=('plan',)
  step 0: next=('triage',)
  step -1: next=('__start__',)

REPLAY from step 2 -> re-runs 'plan' with identical inputs:
  replayed events: ['triaged: task received', 'planned with LLM for refund_order_42']

FORK: patch the state at that checkpoint, branch, and continue:
  forked branch state: {'task': 'refund_order_42', 'risk': 'low', 'events': ['triaged: task received', 'planned with LLM for refund_order_42'], 'plan': 'FORKED PLAN: escalate to fraud team instead'}


## 10. Context-window management — pruning with `RemoveMessage`

Long multi-step workflows accumulate messages until they blow the model's context window. With `add_messages` as the reducer, you prune **declaratively**: send `RemoveMessage(id=...)` updates for messages you want dropped from the persisted state. The audit trail in `events` is untouched — you shrink *model context*, not *history*.

In [10]:
from langgraph.graph.message import RemoveMessage

class ChatState(TypedDict):
    task: str
    messages: Annotated[list, add_messages]           # model-facing context
    events: Annotated[list[str], operator.add]        # audit log (never pruned)

def trim_messages(state: ChatState) -> dict:
    msgs = state["messages"]
    keep = 2
    removed = [RemoveMessage(id=m.id) for m in msgs[:-keep]]
    return {"messages": removed, "events": [f"pruned {len(removed)} old messages"]}

cg = StateGraph(ChatState)
cg.add_node("fill", lambda s: {"messages": [
        {"role": "user", "content": f"step-{i} transcript"} for i in range(1, 6)]})
cg.add_node("trim", trim_messages)
cg.add_edge(START, "fill"); cg.add_edge("fill", "trim"); cg.add_edge("trim", END)

final = cg.compile().invoke({"task": "t", "messages": [], "events": []})
print("messages kept in model context:", len(final["messages"]))
for m in final["messages"]:
    print("   ", m)
print("audit log:", final["events"])

messages kept in model context: 2
    content='step-4 transcript' additional_kwargs={} response_metadata={} id='6fbd09a6-3871-4173-a4fc-0699253e6d8d'
    content='step-5 transcript' additional_kwargs={} response_metadata={} id='c8e05a4c-d058-476d-aa8a-d1435ce20a11'
audit log: ['pruned 3 old messages']


## 11. Async execution — `ainvoke` / `astream`

Async nodes (`async def`) let a single event loop juggle thousands of concurrent workflow threads waiting on I/O — LLM calls, tools, human approvals. `astream(stream_mode="updates")` emits one event per superstep as each node completes: the primitive behind live progress UIs and streaming into an ops dashboard.

In [11]:
import asyncio

async def async_triage(s: RefundState) -> dict:        # imagine: await llm.ainvoke(...)
    await asyncio.sleep(0.01)
    return {"events": ["async triage done"]}

async def async_assess(s: RefundState) -> dict:
    await asyncio.sleep(0.01)
    return {"events": ["async assess done"]}

ag = StateGraph(RefundState)
ag.add_node("triage", async_triage)
ag.add_node("assess", async_assess)
ag.add_edge(START, "triage"); ag.add_edge("triage", "assess"); ag.add_edge("assess", END)
aapp = ag.compile(checkpointer=InMemorySaver())

async def run_async():
    final = await aapp.ainvoke({"task": "order-1", "risk": "low", "decision": "", "events": []},
                               {"configurable": {"thread_id": "async-1"}})
    print("ainvoke result:", final["events"])
    print()
    print("astream('updates') — one event per superstep:")
    async for chunk in aapp.astream({"task": "order-2", "risk": "low", "decision": "", "events": []},
                                    {"configurable": {"thread_id": "async-2"}},
                                    stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"  superstep finished -> node '{node_name}': {update}")

await run_async()

ainvoke result: ['async triage done', 'async assess done']

astream('updates') — one event per superstep:


  superstep finished -> node 'triage': {'events': ['async triage done']}
  superstep finished -> node 'assess': {'events': ['async assess done']}


## Recap — the playbook, now proven in code

| Rule (from the interview answer) | Where you saw it |
|---|---|
| Keep workers **stateless**, externalize state | §2–3: checkpointer + `thread_id`; state survives a fresh connection |
| **Event log**, not just a snapshot | §1: `events: Annotated[list[str], operator.add]` audit trail; §2: checkpoint history |
| **Idempotency** for side effects | §7: `EXECUTION_LEDGER` — duplicate step execution is harmless |
| **Explicit state machine**, async gaps are waiting states | §4: conditional edges as legal transitions; §5: `interrupt()` = durable `WAITING_HUMAN` |
| Handle async gaps deliberately | §5: pause/resume via `Command`; §6: resume after crash with `invoke(None)`; §11: async fan-out |

**Production checkpointer choice:** `InMemorySaver` for tests → `SqliteSaver` for local dev → `PostgresSaver` (or a platform-native store, e.g. Databricks/Temporal-backed) for production. Same code, swap the saver.

**The one-line interview answer:**
> *"I keep workers stateless, checkpoint progress to durable storage after every step as an event log, make all side-effects idempotent, and model the workflow as an explicit state machine — so any step can fail, retry, or wait on a human without losing context or double-executing."*